In [1]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect("developer_project.duckdb")

# Optional display settings for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


### Load Raw Tables

In [2]:
with open("DEVPM_8674_Cal_Poly_Export_dev_activity(4).csv", "r", encoding="utf-8", errors="replace") as f:
    for i in range(10):
        print(f.readline().strip())

activity|activity_date|activity_name|activity_type|activity_role|activity_attendance|dev_contact|activity_score|filepath|activity_id|pk1|pk2|lead_source|nvidia_campaign_id|gtc_nvidia_campaign_id|lead_source_details
DLI Training|2025-11-12T06:39:30.000Z|Building RAG Agents with LLMs|Instructor-Led|Attendee|Attended Live|6417584|40.0||1875418|1875418|course-v1:DLI+S-FX-15+V1-ZH||||
DLI Training|2024-05-18T15:53:18.000Z|Getting Started with Deep Learning|Self-Paced Online|Attendee|Attended Live|6752524|20.0||764222|764222|course-v1:DLI+S-FX-01+V1||||
DLI Training|2024-04-17T09:30:35.000Z|(COURSE ENDING ON 7/1/24) High-Performance Computing with Containers|Self-Paced Online|Attendee|Attended Live|6389272|20.0||736857|736857|course-v1:DLI+L-AC-25+V1-ZH||||
DLI Training|2024-05-22T11:40:34.000Z|Getting Started with Deep Learning|Self-Paced Online|Attendee|Attended Live|6763105|20.0||766590|766590|course-v1:DLI+S-FX-01+V1||||
DLI Training|2024-05-30T08:31:34.000Z|Prompt Engineering with LLaMA

In [3]:
# Activity data
con.execute("""
CREATE OR REPLACE TABLE activity_raw AS
SELECT *
FROM read_csv_auto(
    'DEVPM_8674_Cal_Poly_Export_dev_activity(4).csv',
    delim='|',
    header=True,
    all_varchar=True,
    sample_size=-1,
    max_line_size = 10000000
)
""")

# Contact data
con.execute("""
CREATE OR REPLACE TABLE contact_raw AS
SELECT *
FROM read_csv_auto(
    'DEVPM_8674_Cal_Poly_Export_dev_contact(1).csv',
    delim=',',
    header=True,
    all_varchar=True,
    sample_size=-1
)
""")

# SDK download data
con.execute("""
CREATE OR REPLACE TABLE sdk_download_raw AS
SELECT *
FROM read_csv_auto(
    'DEVPM_8674_Cal_Poly_Export_sdk_download(1).csv',
    delim=',',
    header=True,
    all_varchar=True,
    sample_size=-1
)
""")

100% ▕██████████████████████████████████████▏ (00:00:41.06 elapsed)     
100% ▕██████████████████████████████████████▏ (00:00:10.65 elapsed)     
100% ▕██████████████████████████████████████▏ (00:00:41.83 elapsed)     


### 2. Create cleaned analysis tables


In [4]:
# Activity clean
# Keep both analysis columns and source-tracing columns
con.execute("""
CREATE OR REPLACE TABLE activity_clean AS
SELECT
    dev_contact,
    activity,
    activity_name,
    activity_type,
    activity_role,
    activity_attendance,
    TRY_CAST(activity_score AS DOUBLE) AS activity_score,
    TRY_CAST(activity_date AS TIMESTAMP) AS activity_date,
    activity_id,
    filepath,
    pk1,
    pk2,
    lead_source,
    nvidia_campaign_id,
    gtc_nvidia_campaign_id,
    lead_source_details
FROM activity_raw
""")

# Contact clean
con.execute("""
CREATE OR REPLACE TABLE contact_clean AS
SELECT
    developer_id,
    program_application_source,
    country,
    region,
    sub_region,
    zone,
    territory,
    organization_english_name,
    development_areas,
    other_development_areas,
    industry_segment_vertical,
    other_industry_segment_vertical,
    sub_industry_segment_vertical,
    fields_of_interest,
    other_fields_of_interest,
    TRY_CAST(first_program_application_date AS TIMESTAMP) AS first_program_application_date,
    account_id,
    account_name,
    TRY_CAST(last_activity_date AS TIMESTAMP) AS last_activity_date,
    TRY_CAST(last_modified_date AS TIMESTAMP) AS last_modified_date,
    TRY_CAST(created_date AS TIMESTAMP) AS created_date,
    wwfo_category,
    wwfo_target_list,
    account_industry_segment,
    account_source,
    account_type,
    TRY_CAST(devzone_last_login_date AS TIMESTAMP) AS devzone_last_login_date,
    organization_website,
    inception_id,
    TRY_CAST(first_activity_date AS TIMESTAMP) AS first_activity_date,
    normalized_account_name,
    TRY_CAST(rdp_exit_date AS TIMESTAMP) AS rdp_exit_date
FROM contact_raw
""")

# SDK download clean
con.execute("""
CREATE OR REPLACE TABLE sdk_download_clean AS
SELECT
    source,
    sdk_name,
    PRODUCTNAME AS product_name,
    PRODUCTRELEASE AS product_release,
    country,
    region,
    subregion,
    territory,
    zone,
    TRY_CAST(downloaddate AS DATE) AS download_date,
    FILETYPE AS file_type,
    OPERATINGSYSTEM AS operating_system,
    OS_DISTRIBUTION AS os_distribution,
    ARCHITECTURE AS architecture,
    TRY_CAST(KPI AS DOUBLE) AS kpi,
    TRY_CAST(downloadcount AS BIGINT) AS download_count
FROM sdk_download_raw
""")


100% ▕██████████████████████████████████████▏ (00:00:40.10 elapsed)     
100% ▕██████████████████████████████████████▏ (00:00:11.18 elapsed)     
100% ▕██████████████████████████████████████▏ (00:00:21.25 elapsed)     


### Sanity Checks

In [5]:
print("Tables in database:")
display(con.execute("SHOW TABLES").fetchdf())

print("Activity raw row count:")
display(con.execute("SELECT COUNT(*) AS row_count FROM activity_raw").fetchdf())

print("Contact raw row count:")
display(con.execute("SELECT COUNT(*) AS row_count FROM contact_raw").fetchdf())

print("SDK raw row count:")
display(con.execute("SELECT COUNT(*) AS row_count FROM sdk_download_raw").fetchdf())

print("Activity clean schema:")
display(con.execute("DESCRIBE activity_clean").fetchdf())

print("Contact clean schema:")
display(con.execute("DESCRIBE contact_clean").fetchdf())

print("SDK clean schema:")
display(con.execute("DESCRIBE sdk_download_clean").fetchdf())

Tables in database:


,name
0,activity_clean
1,activity_raw
2,contact_clean
3,contact_raw
4,sdk_download_clean
5,sdk_download_raw


Activity raw row count:


,row_count
0,69347526


Contact raw row count:


...

,column_name,column_type,null,key,default,extra
0,dev_contact,VARCHAR,YES,None,None,None
1,activity,VARCHAR,YES,None,None,None
2,activity_name,VARCHAR,YES,None,None,None
3,activity_type,VARCHAR,YES,None,None,None
4,activity_role,VARCHAR,YES,None,None,None
5,activity_attendance,VARCHAR,YES,None,None,None
6,activity_score,DOUBLE,YES,None,None,None
7,activity_date,TIMESTAMP,YES,None,None,None
8,activity_id,VARCHAR,YES,None,None,None
9,filepath,VARCHAR,YES,None,None,None


Contact clean schema:


,column_name,column_type,null,key,default,extra
0,developer_id,VARCHAR,YES,None,None,None
1,program_application_source,VARCHAR,YES,None,None,None
2,country,VARCHAR,YES,None,None,None
3,region,VARCHAR,YES,None,None,None
4,sub_region,VARCHAR,YES,None,None,None
5,zone,VARCHAR,YES,None,None,None
6,territory,VARCHAR,YES,None,None,None
7,organization_english_name,VARCHAR,YES,None,None,None
8,development_areas,VARCHAR,YES,None,None,None
9,other_development_areas,VARCHAR,YES,None,None,None


SDK clean schema:


,column_name,column_type,null,key,default,extra
0,source,VARCHAR,YES,None,None,None
1,sdk_name,VARCHAR,YES,None,None,None
2,product_name,VARCHAR,YES,None,None,None
3,product_release,VARCHAR,YES,None,None,None
4,country,VARCHAR,YES,None,None,None
5,region,VARCHAR,YES,None,None,None
6,subregion,VARCHAR,YES,None,None,None
7,territory,VARCHAR,YES,None,None,None
8,zone,VARCHAR,YES,None,None,None
9,download_date,DATE,YES,None,None,None


In [6]:
display(con.execute("SHOW TABLES").fetchdf())

,name
0,activity_clean
1,activity_raw
2,contact_clean
3,contact_raw
4,sdk_download_clean
5,sdk_download_raw
